# Basic Functionality

## Import packages and define quantum platform

In [ ]:
import time

import numpy as np
from laboneq._automation.serialization import load_automation_parameters_from_file
from laboneq.simple import *

from laboneq_applications._automation import WorkflowAutomation, WorkflowLayer
from laboneq_applications.experiments import qubit_spectroscopy, ramsey
from laboneq_applications.experiments.options import TuneUpWorkflowOptions
from laboneq_applications.qpu_types.tunable_transmon import demo_platform

In [ ]:
# Create a demonstration QuantumPlatform for a tunable-transmon QPU:
qt_platform = demo_platform(n_qubits=6)

# The platform contains a setup, which is an ordinary LabOne Q DeviceSetup:
setup = qt_platform.setup

# And a tunable-transmon QPU:
qpu = qt_platform.qpu

# Inside the QPU, we have quantum elements, which is a list of six LabOne Q Application
# Library TunableTransmonQubit qubits:
qubits = qpu.quantum_elements

session = Session(setup)
session.connect(do_emulation=True)

## Running experiment workflows manually

### Qubit Spectroscopy

In [ ]:
opts = TuneUpWorkflowOptions()
opts.evaluate = True
opts.update = True
opts

In [ ]:
experiment_workflow = qubit_spectroscopy.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=qubits[0],
    frequencies=np.linspace(6.0e9, 6.3e9, 101),
    options=opts,
)
workflow_result = experiment_workflow.run()

In [ ]:
type(workflow_result)

In [ ]:
workflow_result.tasks["analysis_workflow"].output

In [ ]:
analysis_results = workflow_result.tasks["analysis_workflow"]
fit_results = analysis_results.tasks["fit_data"].output
fit_results["q0"].rsquared

In [ ]:
experiment_workflow.graph.tree

### Ramsey

In [ ]:
opts = TuneUpWorkflowOptions()
opts.evaluate = True
opts.update = True
opts

In [ ]:
experiment_workflow = ramsey.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=qubits[0],
    delays=np.linspace(0, 20e-6, 51),
    detunings=0.67e6,
    options=opts,
)
workflow_result = experiment_workflow.run()

In [ ]:
workflow_result.tasks["analysis_workflow"].output

In [ ]:
analysis_results = workflow_result.tasks["analysis_workflow"]
fit_results = analysis_results.tasks["fit_data"].output
fit_results["q0"].rsquared

In [ ]:
experiment_workflow.graph.tree

## Running experiment workflows using the automation framework

In [ ]:
auto = WorkflowAutomation(
    session,
    qpu=qpu,
    automation_parameters=load_automation_parameters_from_file(
        "basic_functionality.yml"
    ),
    name="example",
)

qs1 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow, ["q0"], key="qs1", depends_on={"root"}
)
auto.add_layer(qs1)
auto.plot();

In [ ]:
auto.run_layer("qs1", node_keys=["q0"])
auto.plot();

In [ ]:
auto["qs1"].status

In [ ]:
r1 = WorkflowLayer(
    ramsey.experiment_workflow, ["q1", "q3"], key="r1", depends_on={"root"}
)
auto.add_layer(r1)

auto.plot();

In [ ]:
auto.plot("layers");

In [ ]:
r2 = WorkflowLayer(
    ramsey.experiment_workflow, ["q0", "q2"], key="r2", depends_on={"qs1"}
)
auto.add_layer(r2)

auto.plot();

In [ ]:
r3 = WorkflowLayer(
    ramsey.experiment_workflow,
    ["q0", "q1", "q2", "q3"],
    key="r3",
    depends_on={"r1"},
)
auto.add_layer(r3)

auto.plot();

In [ ]:
r4 = WorkflowLayer(
    ramsey.experiment_workflow,
    ["q0", "q1", "q2", "q3"],
    key="r4",
    depends_on={"r2", "r3"},
)
auto.add_layer(r4)

auto.plot();

In [ ]:
auto.plot("layers");

In [ ]:
auto["r4_q3"]

In [ ]:
auto.run_layer("r1")
auto.plot();

In [ ]:
auto["r2_q2"].status

In [ ]:
auto.reset()
auto.plot();

In [ ]:
start_time = time.perf_counter()
auto.run()
end_time = time.perf_counter()
auto.plot()
print(f"Time taken (s) = {end_time - start_time}")

In [ ]:
auto.plot("layers");

In [ ]:
auto.remove_layer("r1")
auto.plot();

In [ ]:
auto.plot("layers");

In [ ]:
auto["qs1"]

In [ ]:
auto.add_layer(r1)
auto.add_layer(r3)
auto.add_layer(r4)
auto.plot();

In [ ]:
auto.reset()
auto.run_from_node("qs1_q0")
auto.plot();

In [ ]:
auto["r3"]

In [ ]:
auto.get_layer("r2")

In [ ]:
# print([node for node in auto.nodes(layer_id=1)])
print(list(auto.node_ids(layer_key="qs1")))
# print([layer for layer in auto.layers()])
print(list(auto.layer_keys()))